# Programming Assignment – Adversarial AI in Games

## Project: Briscas with Averaging over Clairvoyance (AoC)

This notebook implements a simplified two-player **Briscas** environment and evaluates three agents:

1. **RandomAgent** – baseline agent that chooses legal cards randomly.
2. **HeuristicAgent** – rule-based agent using simple Briscas strategy.
3. **AoCAgent** – adversarial search agent using **Averaging over Clairvoyance** with minimax and alpha-beta pruning.

The goal is not only to run the game, but also to evaluate the AI using metrics such as:

- Win rate
- 95% confidence interval
- Average score difference
- Standard deviation of score differences
- Wins, losses, and draws


## 0. Create the virtual environment in Anaconda Prompt

Run these commands in **Anaconda Prompt**, not inside Jupyter:

```bash
conda create -n briscas_ai python=3.11 -y
conda activate briscas_ai
pip install numpy gymnasium pettingzoo jupyter pandas matplotlib
python -m ipykernel install --user --name briscas_ai --display-name "Python (briscas_ai)"
jupyter notebook
```

When Jupyter opens, select:

**Kernel → Change Kernel → Python (briscas_ai)**

### Why create a virtual environment?

A virtual environment keeps this project separate from other Python projects. This avoids dependency conflicts and makes the project easier to reproduce.


## 1. Imports

This cell imports the libraries used throughout the notebook.

### Important imports

- `numpy`: numerical arrays and random choices.
- `random`: Python random operations, such as shuffling the deck.
- `copy`: creates deep copies of simulated game states.
- `math`: used for confidence interval calculations.
- `defaultdict`: stores multiple simulated scores per action.
- `pettingzoo.AECEnv`: base class for turn-based multi-agent environments.
- `agent_selector`: helper from PettingZoo for agent ordering.
- `gymnasium.spaces`: defines observation and action spaces.
- `pandas`: organizes evaluation results in a table.


In [1]:
import numpy as np
import random
import copy
import math
from collections import defaultdict

from pettingzoo import AECEnv
from pettingzoo.utils.agent_selector import agent_selector
from gymnasium import spaces

import pandas as pd

## 2. Card definitions

This section defines the Briscas deck.

### Variables

- `SUITS`: the four suits in the Spanish-style deck.
- `RANKS`: the card ranks used in Briscas.
- `POINTS`: how many points each rank is worth.
- `RANK_ORDER`: strength order for deciding which card wins a trick.
- `DECK`: complete 40-card deck represented as tuples `(suit, rank)`.

### Card representation

A card such as `('coins', 1)` means:

- suit: coins
- rank: 1
- points: 11


In [2]:
# Card definitions

SUITS = ['coins', 'cups', 'swords', 'clubs']

# Briscas uses 40 cards: 1, 2, 3, 4, 5, 6, 7, 10, 11, 12 for each suit.
RANKS = [1, 2, 3, 4, 5, 6, 7, 10, 11, 12]

# Point value of each card rank.
POINTS = {
    1: 11,
    3: 10,
    12: 4,
    11: 3,
    10: 2,
    2: 0,
    4: 0,
    5: 0,
    6: 0,
    7: 0
}

# Higher index means stronger card when both cards have the same suit.
RANK_ORDER = {
    r: i for i, r in enumerate([2, 4, 5, 6, 7, 10, 11, 12, 3, 1])
}

# Complete deck of 40 cards.
DECK = [(s, r) for s in SUITS for r in RANKS]


def card_points(card):
    """Return the point value of a card."""
    return POINTS[card[1]]


def trick_winner(lead_card, follow_card, trump_suit):
    """
    Decide who wins a trick.

    Parameters
    ----------
    lead_card : tuple
        Card played first.
    follow_card : tuple
        Card played second.
    trump_suit : str
        Suit that beats non-trump suits.

    Returns
    -------
    int
        0 if the lead card wins.
        1 if the follow card wins.
    """
    lead_suit, lead_rank = lead_card
    follow_suit, follow_rank = follow_card

    # If only the lead card is trump, lead wins.
    if lead_suit == trump_suit and follow_suit != trump_suit:
        return 0

    # If only the follow card is trump, follow wins.
    if follow_suit == trump_suit and lead_suit != trump_suit:
        return 1

    # If both cards have the same suit, stronger rank wins.
    if lead_suit == follow_suit:
        return 0 if RANK_ORDER[lead_rank] >= RANK_ORDER[follow_rank] else 1

    # If no trump and different suits, the lead card wins.
    return 0

## 3. Briscas environment

This class defines the game environment using the PettingZoo AEC style.

AEC means **Agent Environment Cycle**. In simple terms, the environment controls whose turn it is.

### Important environment variables

- `possible_agents`: the players in the game.
- `card_to_idx`: maps each card to an integer action from 0 to 39.
- `idx_to_card`: maps an integer action back to a card.
- `observation_spaces`: defines the vector seen by each agent.
- `action_spaces`: defines 40 possible actions, one per card.
- `hands`: current cards in each player's hand.
- `draw_pile`: remaining deck after initial cards.
- `trump_suit`: suit that wins over non-trump suits.
- `scores`: points accumulated by each player.
- `trick_cards`: cards currently played in the active trick.
- `seen_cards`: cards already played in completed tricks.
- `lead_player`: player who starts the current trick.
- `agent_selection`: player whose turn it is now.

### Observation vector

The observation vector has length:

```text
40 + 41 + 4 + 40 + 1 + 1 = 127
```

It contains:

1. Player hand: 40 values.
2. Opponent card in current trick or no-card flag: 41 values.
3. Trump suit: 4 values.
4. Seen cards: 40 values.
5. Player score normalized by 120.
6. Draw pile size normalized by 34.


In [3]:
class BriscasEnv(AECEnv):
    """Simplified two-player Briscas environment."""

    metadata = {'name': 'briscas_v0'}

    def __init__(self):
        super().__init__()

        # Two-player version.
        self.possible_agents = ['player_0', 'player_1']

        # Convert between cards and action indexes.
        self.card_to_idx = {card: i for i, card in enumerate(DECK)}
        self.idx_to_card = dict(enumerate(DECK))

        # Observation size:
        # 40 hand + 41 opponent/trick info + 4 trump + 40 seen cards + score + draw pile size
        obs_size = 40 + 41 + 4 + 40 + 1 + 1

        self.observation_spaces = {
            agent: spaces.Box(low=0, high=120, shape=(obs_size,), dtype=np.float32)
            for agent in self.possible_agents
        }

        # 40 possible card actions.
        self.action_spaces = {
            agent: spaces.Discrete(40)
            for agent in self.possible_agents
        }

    def reset(self, seed=None, options=None):
        """Start a new game."""

        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)

        deck = list(DECK)
        random.shuffle(deck)

        # Deal 3 cards to each player.
        self.hands = {
            'player_0': deck[:3],
            'player_1': deck[3:6]
        }

        # Remaining cards after initial deal.
        self.draw_pile = deck[6:]

        # Trump suit is based on the first card of the draw pile.
        self.trump_suit = deck[6][0]

        self.scores = {
            'player_0': 0,
            'player_1': 0
        }

        self.trick_cards = {}
        self.seen_cards = set()

        self.lead_player = 'player_0'

        self.agents = list(self.possible_agents)
        self._agent_selector = agent_selector(self.agents)
        self.agent_selection = self.lead_player

        self.rewards = {agent: 0 for agent in self.agents}
        self.terminations = {agent: False for agent in self.agents}
        self.truncations = {agent: False for agent in self.agents}
        self.infos = {agent: {} for agent in self.agents}
        self._cumulative_rewards = {agent: 0 for agent in self.agents}

    def step(self, action):
        """Apply one card action for the current player."""

        agent = self.agent_selection
        card = self.idx_to_card[action]

        # Remove the selected card from the player's hand.
        self.hands[agent].remove(card)

        # Save the card in the current trick.
        self.trick_cards[agent] = card

        opponent = self._opp(agent)

        # If both players have played, resolve the trick.
        if opponent in self.trick_cards:
            lead_card = self.trick_cards[self.lead_player]

            # Determine who played second.
            follow_player = opponent if self.lead_player == agent else agent
            follow_card = self.trick_cards[follow_player]

            winner_index = trick_winner(lead_card, follow_card, self.trump_suit)
            winner = self.lead_player if winner_index == 0 else follow_player
            loser = follow_player if winner_index == 0 else self.lead_player

            # Add trick points to winner.
            self.scores[winner] += card_points(lead_card) + card_points(follow_card)

            # Mark trick cards as seen.
            self.seen_cards.update(self.trick_cards.values())

            # Clear trick.
            self.trick_cards = {}

            # Winner leads next trick.
            self.lead_player = winner

            # Winner draws first, then loser draws.
            for player in [winner, loser]:
                if self.draw_pile:
                    self.hands[player].append(self.draw_pile.pop(0))

            # Game ends when both players have no cards.
            if not self.hands['player_0'] and not self.hands['player_1']:
                self._end_game()
                return

            self.agent_selection = winner

        else:
            # If only one card has been played, opponent plays next.
            self.agent_selection = opponent

        self.rewards = {agent: 0 for agent in self.agents}
        self._accumulate_rewards()

    def _end_game(self):
        """Assign final rewards and terminate the game."""

        score_0 = self.scores['player_0']
        score_1 = self.scores['player_1']

        if score_0 > score_1:
            self.rewards = {'player_0': 1, 'player_1': -1}
        elif score_1 > score_0:
            self.rewards = {'player_0': -1, 'player_1': 1}
        else:
            self.rewards = {'player_0': 0, 'player_1': 0}

        self.terminations = {agent: True for agent in self.agents}
        self._accumulate_rewards()
        self.agents = []

    def observe(self, agent):
        """Return the observation vector for a player."""

        # Own hand vector.
        hand_vec = np.zeros(40, dtype=np.float32)
        for card in self.hands.get(agent, []):
            hand_vec[self.card_to_idx[card]] = 1

        opponent = self._opp(agent)

        # Opponent card currently visible in the trick.
        opp_vec = np.zeros(41, dtype=np.float32)
        if opponent in self.trick_cards:
            opp_vec[self.card_to_idx[self.trick_cards[opponent]] + 1] = 1
        else:
            opp_vec[0] = 1

        # Trump suit one-hot vector.
        trump_vec = np.zeros(4, dtype=np.float32)
        trump_vec[SUITS.index(self.trump_suit)] = 1

        # Cards already seen.
        seen_vec = np.zeros(40, dtype=np.float32)
        for card in self.seen_cards:
            seen_vec[self.card_to_idx[card]] = 1

        # Normalized score and draw pile size.
        score_feature = [self.scores.get(agent, 0) / 120.0]
        draw_pile_feature = [len(self.draw_pile) / 34.0]

        return np.concatenate([
            hand_vec,
            opp_vec,
            trump_vec,
            seen_vec,
            score_feature,
            draw_pile_feature
        ])

    def action_mask(self, agent):
        """Return which card actions are legal for a player."""

        mask = np.zeros(40, dtype=np.int8)

        for card in self.hands.get(agent, []):
            mask[self.card_to_idx[card]] = 1

        return mask

    def observation_space(self, agent):
        return self.observation_spaces[agent]

    def action_space(self, agent):
        return self.action_spaces[agent]

    def render(self):
        pass

    def close(self):
        pass

    def _opp(self, agent):
        """Return the opponent of the current player."""
        return 'player_1' if agent == 'player_0' else 'player_0'

## 4. Baseline agents

This section defines the basic agents used for comparison.

### RandomAgent

Chooses a random legal card. This is important because the assignment expects testing against random or naive agents.

### HeuristicAgent

Uses simple rules:

1. If leading a trick, play the lowest non-trump card when possible.
2. If responding to a valuable opponent card, try to win with the lowest winning card.
3. If winning is not worth it or not possible, play the lowest card.

### Important variables

- `agent_id`: identifies whether the agent is `player_0` or `player_1`.
- `legal`: legal actions from the action mask.
- `hand`: cards currently held by the agent.
- `trump`: trump suit.
- `opp_card`: card played by the opponent in the current trick.
- `winners`: cards in hand that can beat the opponent card.


In [4]:
class RandomAgent:
    """Baseline agent that chooses a legal card randomly."""

    def __init__(self, agent_id):
        self.agent_id = agent_id

    def act(self, env):
        legal = np.where(env.action_mask(self.agent_id) == 1)[0]
        return int(np.random.choice(legal))


class HeuristicAgent:
    """Simple rule-based Briscas agent."""

    def __init__(self, agent_id):
        self.agent_id = agent_id

    def act(self, env):
        hand = env.hands[self.agent_id]
        trump = env.trump_suit
        opponent = env._opp(self.agent_id)

        # Card value used to choose the lowest useful card.
        value = lambda card: (card_points(card), RANK_ORDER[card[1]])

        # Case 1: agent leads the trick.
        if self.agent_id == env.lead_player and opponent not in env.trick_cards:
            non_trumps = [card for card in hand if card[0] != trump]
            return env.card_to_idx[min(non_trumps if non_trumps else hand, key=value)]

        # Case 2: agent responds to opponent card.
        opp_card = env.trick_cards.get(opponent)
        trick_points = card_points(opp_card) if opp_card else 0

        # Cards that can win the trick.
        winners = [
            card for card in hand
            if opp_card and trick_winner(opp_card, card, trump) == 1
        ]

        # Prefer winning with a non-trump if the trick has meaningful points.
        non_trump_winners = [card for card in winners if card[0] != trump]
        if non_trump_winners and trick_points >= 5:
            return env.card_to_idx[min(non_trump_winners, key=value)]

        # If needed, win with trump.
        trump_winners = [card for card in winners if card[0] == trump]
        if trump_winners and trick_points >= 5:
            return env.card_to_idx[min(trump_winners, key=value)]

        # Otherwise discard the lowest card.
        return env.card_to_idx[min(hand, key=value)]

## 5. AoC Agent: Averaging over Clairvoyance

This is the main AI agent.

The opponent's hand is partially unknown. The AoC agent handles this by:

1. Identifying unknown cards.
2. Sampling possible opponent hands.
3. For each possible hidden state, simulating actions using minimax.
4. Averaging the value of each possible move.
5. Choosing the move with the best average score.

### Parameters

- `agent_id`: player controlled by this agent.
- `n_samples`: number of possible hidden states sampled.
  - Higher value = better approximation.
  - Higher value = slower execution.
- `depth`: how many future card plays the minimax search explores.
  - Higher value = stronger lookahead.
  - Higher value = slower execution.

### Important variables

- `my_hand`: cards available to the AoC agent.
- `known`: cards known to the agent.
- `unknown`: cards that may be in the opponent hand or draw pile.
- `sampled_opp`: sampled possible opponent hand.
- `remaining`: simulated draw pile after assigning opponent hand.
- `state`: simulated complete state used for minimax.
- `action_scores`: stores the minimax scores for each possible action.
- `alpha`, `beta`: alpha-beta pruning bounds.
- `maximizing`: whether the current minimax level benefits the AoC agent.


In [7]:
class AoCAgent:
    """Averaging over Clairvoyance agent using minimax and alpha-beta pruning."""

    def __init__(self, agent_id, n_samples=15, depth=4):
        self.agent_id = agent_id
        self.n_samples = n_samples
        self.depth = depth

    def act(self, env):
        """Choose a card by averaging minimax evaluations over sampled hidden states."""

        my_hand = list(env.hands[self.agent_id])
        opponent_id = env._opp(self.agent_id)

        # Cards known by this agent.
        known = set(my_hand) | env.seen_cards | set(env.trick_cards.values())

        # Cards not known. These may belong to opponent or draw pile.
        unknown = [card for card in DECK if card not in known]

        opponent_hand_size = len(env.hands[opponent_id])

        # Dictionary action -> list of scores across samples.
        action_scores = defaultdict(list)

        for _ in range(self.n_samples):
            # Sample a possible opponent hand from unknown cards.
            sampled_opponent_hand = random.sample(
                unknown,
                min(opponent_hand_size, len(unknown))
            )

            # Remaining unknown cards become a simulated draw pile.
            remaining = [
                card for card in unknown
                if card not in sampled_opponent_hand
            ]
            random.shuffle(remaining)

            # Simulated complete state.
            state = {
                'hands': {
                    self.agent_id: list(my_hand),
                    opponent_id: sampled_opponent_hand
                },
                'draw_pile': list(remaining),
                'trump_suit': env.trump_suit,
                'scores': dict(env.scores),
                'trick_cards': dict(env.trick_cards),
                'lead_player': env.lead_player,
                'to_play': self.agent_id
            }

            # Evaluate each legal card from the current hand.
            for card in my_hand:
                next_state = self._play_card(state, self.agent_id, card)

                score = self._minimax(
                    next_state,
                    self.depth - 1,
                    alpha=-9999,
                    beta=9999,
                    maximizing=(next_state['to_play'] == self.agent_id)
                )

                action_scores[env.card_to_idx[card]].append(score)

        # Pick the card with the best average score.
        return max(action_scores, key=lambda action: np.mean(action_scores[action]))

    def _minimax(self, state, depth, alpha, beta, maximizing):
        """Minimax search with alpha-beta pruning."""

        hand = state['hands'].get(state['to_play'], [])

        # Stop if depth is reached or current player has no cards.
        if depth == 0 or not hand:
            opponent = self._opp(self.agent_id)
            return state['scores'][self.agent_id] - state['scores'][opponent]

        if maximizing:
            best = -9999

            for card in hand:
                next_state = self._play_card(state, state['to_play'], card)

                value = self._minimax(
                    next_state,
                    depth - 1,
                    alpha,
                    beta,
                    maximizing=(next_state['to_play'] == self.agent_id)
                )

                best = max(best, value)
                alpha = max(alpha, best)

                # Alpha-beta pruning.
                if beta <= alpha:
                    break

            return best

        else:
            best = 9999

            for card in hand:
                next_state = self._play_card(state, state['to_play'], card)

                value = self._minimax(
                    next_state,
                    depth - 1,
                    alpha,
                    beta,
                    maximizing=(next_state['to_play'] == self.agent_id)
                )

                best = min(best, value)
                beta = min(beta, best)

                # Alpha-beta pruning.
                if beta <= alpha:
                    break

            return best

    def _play_card(self, state, player, card):
        """Simulate playing a card in a copied state."""

        simulated_state = copy.deepcopy(state)

        simulated_state['hands'][player].remove(card)
        simulated_state['trick_cards'][player] = card

        opponent = self._opp(player)

        # If both players have played in this trick, resolve trick.
        if opponent in simulated_state['trick_cards']:
            lead_card = simulated_state['trick_cards'][simulated_state['lead_player']]

            follow_player = opponent if simulated_state['lead_player'] == player else player
            follow_card = simulated_state['trick_cards'][follow_player]

            winner_index = trick_winner(
                lead_card,
                follow_card,
                simulated_state['trump_suit']
            )

            winner = simulated_state['lead_player'] if winner_index == 0 else follow_player
            loser = follow_player if winner_index == 0 else simulated_state['lead_player']

            simulated_state['scores'][winner] += card_points(lead_card) + card_points(follow_card)

            simulated_state['trick_cards'] = {}
            simulated_state['lead_player'] = winner

            # Winner draws first.
            for player_id in [winner, loser]:
                if simulated_state['draw_pile']:
                    simulated_state['hands'][player_id].append(
                        simulated_state['draw_pile'].pop(0)
                    )

            simulated_state['to_play'] = winner

        else:
            simulated_state['to_play'] = opponent

        return simulated_state

    def _opp(self, player):
        """Return opponent player id."""
        return 'player_1' if player == 'player_0' else 'player_0'

## 6. Game runner

This function runs one full game.

### Parameters

- `agent_0`: object controlling `player_0`.
- `agent_1`: object controlling `player_1`.
- `seed`: optional random seed for reproducibility.

### Returns

A dictionary with:

- `winner`: `0`, `1`, or `'draw'`.
- `score_0`: final score for player 0.
- `score_1`: final score for player 1.
- `score_diff`: `score_0 - score_1`.


In [8]:
def run_game(agent_0, agent_1, seed=None):
    """Run one complete game between two agents."""

    env = BriscasEnv()
    env.reset(seed=seed)

    agents_map = {
        'player_0': agent_0,
        'player_1': agent_1
    }

    while env.agents:
        current_agent = env.agent_selection
        action = agents_map[current_agent].act(env)
        env.step(action)

    score_0 = env.scores['player_0']
    score_1 = env.scores['player_1']

    if score_0 > score_1:
        winner = 0
    elif score_1 > score_0:
        winner = 1
    else:
        winner = 'draw'

    return {
        'winner': winner,
        'score_0': score_0,
        'score_1': score_1,
        'score_diff': score_0 - score_1
    }

## 7. Test one game

Run one game to verify that the environment and agents work before running many experiments.


In [9]:
test_result = run_game(
    AoCAgent('player_0', n_samples=5, depth=3),
    RandomAgent('player_1'),
    seed=0
)

test_result

{'winner': 0, 'score_0': 84, 'score_1': 36, 'score_diff': 48}

## 8. Evaluation Function

This function runs many games for a matchup.

### Why switch player positions?

The first player may have an advantage. To make the comparison fair:

- First half: Agent A is `player_0`.
- Second half: Agent A is `player_1`.

### Parameters

- `agent_a_cls`: class of Agent A, the first agent in the matchup label.
- `agent_b_cls`: class of Agent B, the second agent in the matchup label.
- `n_games`: number of games to run.
- `a_kw`: optional parameters for Agent A.
- `b_kw`: optional parameters for Agent B.
- `label`: name of the matchup.

### Metrics

- `win_rate`: proportion of games won by Agent A.
- `ci_95_low`: lower bound of the 95% confidence interval for Agent A's win rate.
- `ci_95_high`: upper bound of the 95% confidence interval for Agent A's win rate.
- `avg_diff`: average score difference from Agent A's perspective.
- `std_diff`: standard deviation of score differences.
- `wins_a`: number of wins by Agent A.
- `wins_b`: number of wins by Agent B.
- `draws`: number of draws.
- `n_games`: total number of games played.

In [10]:
def evaluate_matchup(agent_a_cls, agent_b_cls, n_games=500, a_kw=None, b_kw=None, label=''):
    """Evaluate Agent A against Agent B across many games."""

    a_kw = a_kw or {}
    b_kw = b_kw or {}

    wins_a = 0
    wins_b = 0
    draws = 0
    diffs = []

    half = n_games // 2

    for i in range(n_games):

        # First half: Agent A is player_0.
        if i < half:
            result = run_game(
                agent_a_cls('player_0', **a_kw),
                agent_b_cls('player_1', **b_kw),
                seed=i
            )

            won_a = result['winner'] == 0
            won_b = result['winner'] == 1
            diff = result['score_diff']

        # Second half: Agent A is player_1.
        else:
            result = run_game(
                agent_b_cls('player_0', **b_kw),
                agent_a_cls('player_1', **a_kw),
                seed=i
            )

            won_a = result['winner'] == 1
            won_b = result['winner'] == 0

            # Reverse score difference so it is still from Agent A's perspective.
            diff = -result['score_diff']

        if won_a:
            wins_a += 1
        elif won_b:
            wins_b += 1
        else:
            draws += 1

        diffs.append(diff)

    # Wilson 95% confidence interval for win rate.
    p = wins_a / n_games
    n = n_games
    z = 1.96

    ci_low = (
        p + z**2 / (2*n)
        - z * math.sqrt((p*(1-p) + z**2/(4*n)) / n)
    ) / (1 + z**2/n)

    ci_high = (
        p + z**2 / (2*n)
        + z * math.sqrt((p*(1-p) + z**2/(4*n)) / n)
    ) / (1 + z**2/n)

    return {
        'label': label,
        'win_rate': round(p, 4),
        'ci_95_low': round(ci_low, 4),
        'ci_95_high': round(ci_high, 4),
        'avg_diff': round(float(np.mean(diffs)), 2),
        'std_diff': round(float(np.std(diffs)), 2),
        'wins_a': wins_a,
        'wins_b': wins_b,
        'draws': draws,
        'n_games': n_games
    }

## 9. Run experiments

For a first quick test, use `N_GAMES = 50`.

For the final report, use at least `N_GAMES = 300` or `N_GAMES = 500` if your computer can run it in a reasonable time.

### Main parameters

- `N_GAMES`: total games per matchup.
- `AOC_KW`: parameters for AoC agent.
  - `n_samples`: number of hidden-state samples.
  - `depth`: minimax search depth.


In [11]:
N_GAMES = 50

AOC_KW = {
    'n_samples': 15,
    'depth': 4
}

matchups = [
    (AoCAgent, RandomAgent, AOC_KW, {}, 'AoC vs Random'),
    (AoCAgent, HeuristicAgent, AOC_KW, {}, 'AoC vs Heuristic'),
    (HeuristicAgent, RandomAgent, {}, {}, 'Heuristic vs Random')
]

results = []

for agent_a, agent_b, a_kw, b_kw, label in matchups:
    print(f'Running: {label}')
    result = evaluate_matchup(
        agent_a,
        agent_b,
        n_games=N_GAMES,
        a_kw=a_kw,
        b_kw=b_kw,
        label=label
    )
    results.append(result)

results_df = pd.DataFrame(results)
results_df

Running: AoC vs Random
Running: AoC vs Heuristic
Running: Heuristic vs Random


,label,win_rate,ci_95_low,ci_95_high,avg_diff,std_diff,wins_a,wins_b,draws,n_games
0,AoC vs Random,0.92,0.8116,0.9685,46.60,33.27,46,4,0,50
1,AoC vs Heuristic,0.58,0.4423,0.7063,7.96,41.45,29,21,0,50
2,Heuristic vs Random,0.82,0.6920,0.9023,34.16,36.86,41,8,1,50


## 10. Export results to CSV

This is useful for including the results in the report.


In [10]:
results_df.to_csv('briscas_ai_results.csv', index=False)
print('Saved results to briscas_ai_results.csv')

Saved results to briscas_ai_results.csv


## 12. Explanation for the report

### Chosen game

The selected game is Briscas, a two-player adversarial card game with hidden information because each player cannot directly observe the opponent's hand or the order of the draw pile.

### Chosen approach

The selected approach is Averaging over Clairvoyance. Since the game is partially observable, the agent samples possible hidden states of the game, assumes each sampled state is fully observable, evaluates possible actions using minimax with alpha-beta pruning, and then averages the results across samples.

### Why PettingZoo?

PettingZoo is appropriate because it is designed for multi-agent environments. The AEC structure fits turn-based games where one agent acts at a time.

### Agents Implemented

Three agents were used in the evaluation:

1. **RandomAgent**  
   This agent selects a legal card randomly. It works as the simplest baseline and helps verify whether the AI performs better than chance.

2. **HeuristicAgent**  
   This agent follows simple Briscas rules. For example, it tries to avoid wasting trump cards, plays low-value cards when leading, and attempts to win valuable tricks when appropriate.

3. **AoCAgent**  
   This is the main AI agent. It uses Averaging over Clairvoyance by sampling possible opponent hands, simulating future moves with minimax, and selecting the action with the best average result.


### Evaluation Design

The AI was evaluated by running multiple games between different agents. The main matchups were:

1. **AoC vs Random**
2. **AoC vs Heuristic**
3. **Heuristic vs Random**

The purpose of these experiments was to compare the main AI agent against a weak baseline and a stronger rule-based opponent. The comparison between Heuristic and Random was also included to confirm that the heuristic agent is stronger than a random strategy.

The evaluation reports the following metrics:

- **Win rate:** percentage of games won by the first agent in the matchup.
- **95% confidence interval:** estimated range where the true win rate is likely to fall.
- **Average score difference:** average point advantage of the first agent.
- **Standard deviation of score difference:** how much the score differences varied across games.
- **Wins, losses, and draws:** total outcomes across all games.
  
### Results Summary

The experimental results show that the AoC agent performed strongly against the random baseline. In the **AoC vs Random** matchup, AoC won **46 out of 50 games**, achieving a **92% win rate**. This indicates that AoC makes much better decisions than a random player.

Against the heuristic agent, the result was more competitive. In the **AoC vs Heuristic** matchup, AoC won **29 out of 50 games**, achieving a **58% win rate**. This suggests that the heuristic agent is a stronger opponent than the random baseline, but AoC still performed slightly better overall.

The **Heuristic vs Random** matchup also confirms that the heuristic strategy is stronger than random play. The heuristic agent won **41 out of 50 games**, achieving an **82% win rate**.

Overall, these results suggest that the AoC agent is the strongest agent among the three tested strategies, followed by the heuristic agent, and then the random agent.

### Limitations

The environment is a simplified version of Briscas. The AoC agent does not know the opponent's exact hand, so it approximates the hidden information by sampling possible opponent hands. Increasing `n_samples` and `depth` can improve decision quality but increases computation time.
